In [1]:
!pip install -q fastapi uvicorn python-multipart nest_asyncio pyngrok transformers peft accelerate bitsandbytes sentence-transformers faiss-cpu rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 86.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 2.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━

In [2]:
import os
import json
import random
import torch
import socket
import threading
import time
import requests
import re
import pickle
import numpy as np
import faiss
from typing import Optional, List, Dict
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok, conf
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sentence_transformers import SentenceTransformer, CrossEncoder

nest_asyncio.apply()

2025-12-12 13:24:08.522028: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765545848.764245      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765545848.820551      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
class VietnameseRAGSystem:
    def __init__(self, embedding_model_name="intfloat/multilingual-e5-large-instruct", cross_encoder_name="namdp-ptit/ViRanker", device=None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = device
        self.embedding_model = SentenceTransformer(embedding_model_name, device=device)
        self.embedding_model.eval()
        self.cross_encoder = CrossEncoder(cross_encoder_name, device=device)
        self.index = None
        self.chunks = []
        self.metadata = []
        
    def load_index(self, faiss_file, metadata_file):
        self.index = faiss.read_index(faiss_file)
        with open(metadata_file, 'rb') as f:
            index_data = pickle.load(f)
        self.chunks = index_data['chunks']
        self.metadata = index_data['metadata']

rag_system = VietnameseRAGSystem()
faiss_cache_file = "/kaggle/input/vectordb/rag_index.faiss"
metadata_cache_file = "/kaggle/input/vectordb/rag_metadata.pkl"
rag_system.load_index(faiss_cache_file, metadata_cache_file)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [4]:
model_id = "Qwen/Qwen3-4B"
adapter_dir = "/kaggle/input/qwen-finetuned/transformers/default/3/qwen_finetuned"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
tokenizer = AutoTokenizer.from_pretrained(adapter_dir, use_fast=True, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16, use_cache=True, low_cpu_mem_usage=True)
model = PeftModel.from_pretrained(base_model, adapter_dir)
model = model.merge_and_unload()
model.eval()
device = next(model.parameters()).device

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [5]:
TOPIC_RANGES = {
    1: {"name": "Lịch Sử Việt Nam Tập 1: Từ khởi thủy đến thế kỷ X", "start": 0, "end": 1195},
    2: {"name": "Lịch Sử Việt Nam Tập 2: Từ thế kỷ X đến thế kỷ XIV", "start": 1196, "end": 2553},
    3: {"name": "Lịch Sử Việt Nam Tập 3: Từ thế kỷ XV đến thế kỷ XVI", "start": 2554, "end": 4195},
    4: {"name": "Lịch Sử Việt Nam Tập 4: Từ thế kỷ XVII đến thế kỷ XVIII", "start": 4196, "end": 5414},
    5: {"name": "Lịch Sử Việt Nam Tập 5: Từ năm 1802 đến năm 1858", "start": 5415, "end": 7025},
    6: {"name": "Lịch Sử Việt Nam Tập 6: Từ năm 1859 đến năm 1896", "start": 7026, "end": 7911},
    7: {"name": "Lịch Sử Việt Nam Tập 7: Từ năm 1897 đến năm 1918", "start": 7912, "end": 9278},
    8: {"name": "Lịch Sử Việt Nam Tập 8: Từ năm 1919 đến năm 1930", "start": 9279, "end": 10569},
    9: {"name": "Lịch Sử Việt Nam Tập 9: Từ năm 1930 đến năm 1945", "start": 10570, "end": 12201},
    10: {"name": "Lịch Sử Việt Nam Tập 10: Từ năm 1945 đến năm 1950", "start": 12202, "end": 13658},
    11: {"name": "Lịch Sử Việt Nam Tập 11: Từ năm 1951 đến năm 1954", "start": 13659, "end": 14797},
    12: {"name": "Lịch Sử Việt Nam Tập 12: Từ năm 1954 đến năm 1965", "start": 14798, "end": 16041},
    13: {"name": "Lịch Sử Việt Nam Tập 13: Từ năm 1965 đến năm 1975", "start": 16042, "end": 17441},
    14: {"name": "Lịch Sử Việt Nam Tập 14: Từ năm 1975 đến năm 1986", "start": 17442, "end": 18585},
    15: {"name": "Lịch Sử Việt Nam Tập 15: Từ năm 1986 đến năm 2000", "start": 18586, "end": 19508}
}

In [10]:
import re
import random
import torch
import json
from typing import Optional, List, Dict

class BatchQuizGenerator:
    def __init__(self, rag_system, tokenizer, model, device):
        self.rag_system = rag_system
        self.tokenizer = tokenizer
        self.model = model
        self.device = device

    def get_valid_indices(self, topic_ids: List[int]):
        all_chunks = self.rag_system.chunks
        candidate_indices = []
        
        if topic_ids and len(topic_ids) > 0:
            for tid in topic_ids:
                if tid in TOPIC_RANGES:
                    topic_range = TOPIC_RANGES[tid]
                    candidate_indices.extend(range(topic_range["start"], min(topic_range["end"] + 1, len(all_chunks))))
            if not candidate_indices: 
                candidate_indices = list(range(len(all_chunks)))
        else:
            candidate_indices = list(range(len(all_chunks)))
            
        return candidate_indices

    def get_single_random_context(self, candidate_indices):
        idx = random.choice(candidate_indices)
        all_chunks = self.rag_system.chunks
        all_metadata = self.rag_system.metadata
        
        return {
            "text": all_chunks[idx]["text"],
            "metadata": all_metadata[idx],
            "index": idx
        }

    def get_topic_id_from_index(self, index):
        for topic_id, topic_info in TOPIC_RANGES.items():
            if topic_info["start"] <= index <= topic_info["end"]:
                return topic_id
        return None

    def clean_text(self, text):
        if not text: return ""
        text = re.sub(r'\*\*|__', '', text)
        
        patterns = [
            r'^(Theo|Dựa vào|Căn cứ vào) (đoạn văn|tư liệu|văn bản|nguồn|bài viết|tác giả).*?([,;:]\s*|là\s*)',
            r'^Đoạn (văn|tư liệu) (cho biết|đề cập|nói về).*?([,;:]\s*|là\s*)',
            r'^Trong (đoạn văn|tư liệu).*?([,;:]\s*|là\s*)',
            r'^(Câu hỏi|Trả lời):',
        ]
        
        for pattern in patterns:
            text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.MULTILINE)
            
        text = text.strip()
        if text:
            text = text[0].upper() + text[1:]
            
        return text

    def construct_single_prompt(self, context):
        period = context['metadata'].get('trieu_dai', 'Lịch sử Việt Nam')
        
        prompt_content = f"""<|im_start|>system
Bạn là một chuyên gia biên soạn đề thi Lịch sử Việt Nam chuyên nghiệp. Nhiệm vụ của bạn là tạo ra câu hỏi trắc nghiệm khách quan chất lượng cao.

TUÂN THỦ TUYỆT ĐỐI CÁC QUY TẮC SAU:
1. **VĂN PHONG TỰ NHIÊN**: Câu hỏi phải đi thẳng vào vấn đề (Ví dụ: "Chiến thắng Bạch Đằng diễn ra năm nào?"). 
   - TUYỆT ĐỐI KHÔNG dùng các cụm từ: "Theo đoạn văn", "Dựa vào tư liệu", "Đoạn văn trên nói về", "Tác giả cho biết".
   - Nếu vi phạm quy tắc này, câu hỏi sẽ bị loại bỏ.
2. **TÍNH CHÍNH XÁC**: Đáp án đúng và Lời giải thích phải được suy luận LOGIC trực tiếp từ [NGỮ LIỆU]. Không bịa đặt thông tin bên ngoài.
3. **CẤU TRÚC**:
   - 4 Lựa chọn (A, B, C, D).
   - Chỉ có 1 đáp án đúng duy nhất.
   - 3 phương án nhiễu phải nghe có vẻ hợp lý (về mặt lịch sử) nhưng sai so với ngữ liệu.
4. **ĐỊNH DẠNG**: Trả về DUY NHẤT một JSON Object hợp lệ, không kèm lời dẫn, không markdown block.

<|im_end|>
<|im_start|>user
[NGỮ LIỆU THAM KHẢO - Bối cảnh: {period}]
{context['text']}

[YÊU CẦU OUTPUT]
Hãy tạo 1 câu hỏi trắc nghiệm từ ngữ liệu trên theo format JSON dưới đây:
{{
  "question": "Nội dung câu hỏi (Ngắn gọn, trực tiếp, không trích dẫn nguồn)",
  "options": [
    "A. Lựa chọn 1", 
    "B. Lựa chọn 2", 
    "C. Lựa chọn 3", 
    "D. Lựa chọn 4"
  ],
  "correct_answer": "A",
  "explanation": "Giải thích ngắn gọn nguyên nhân đúng/sai dựa trên dữ kiện ngữ liệu."
}}
<|im_end|>
<|im_start|>assistant
"""
        return prompt_content

    def generate_stream(self, num_questions_needed, topic_ids: List[int]):
        current_count = 0
        candidate_indices = self.get_valid_indices(topic_ids)
        
        retry_count = 0
        max_retries = num_questions_needed * 3
        
        while current_count < num_questions_needed and retry_count < max_retries:
            context = self.get_single_random_context(candidate_indices)
            
            if len(context['text']) < 50:
                retry_count += 1
                continue

            prompt_text = self.construct_single_prompt(context)
            
            inputs = self.tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=3072).to(self.device)

            with torch.inference_mode():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=512, 
                    do_sample=True,
                    temperature=0.4,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    repetition_penalty=1.1,
                )

            input_length = inputs.input_ids.shape[1]
            response_text = self.tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
            
            quiz_item = self.parse_single_json_response(response_text, context)
            
            if quiz_item:
                current_count += 1
                yield {
                    "generated_count": current_count,
                    "total_requested": num_questions_needed,
                    "batch_data": [quiz_item] 
                }
            else:
                retry_count += 1
                continue

    def parse_single_json_response(self, text, context):
        try:
            json_start = text.find('{')
            json_end = text.rfind('}') + 1
            
            if json_start != -1 and json_end != -1:
                json_str = text[json_start:json_end]
                item = json.loads(json_str)
                
                required_keys = ["question", "options", "correct_answer", "explanation"]
                if all(key in item for key in required_keys) and len(item["options"]) >= 4:
                    
                    q_text = self.clean_text(item["question"])
                    
                    if len(q_text) < 10: return None
                    
                    item["question"] = q_text
                    item["explanation"] = self.clean_text(item.get("explanation", "Dựa trên ngữ liệu."))
                    item["options"] = [self.clean_text(opt) for opt in item["options"][:4]]
                    
                    item["source_id"] = context["index"]
                    
                    return item
        except json.JSONDecodeError:
            return None
        except Exception:
            return None
        return None

quiz_generator = BatchQuizGenerator(rag_system, tokenizer, model, device)

In [11]:
quiz_generator = BatchQuizGenerator(rag_system, tokenizer, model, device)

In [12]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import Optional, List, Dict
import json
import asyncio

app = FastAPI()

class QuizRequest(BaseModel):
    num_questions: int = 5
    topic_ids: Optional[List[int]] = [] 
    session_id: Optional[str] = None

@app.get("/")
async def root():
    return {"message": "Vietnamese History Quiz Generator API", "version": "3.1.0"}

@app.get("/health")
async def health_check():
    return {"status": "healthy", "rag_system_ready": len(rag_system.chunks) > 0, "model_ready": model is not None}

@app.get("/topics")
async def get_topics():
    topics = []
    for topic_id, topic_info in TOPIC_RANGES.items():
        topic_data = {
            "id": topic_id, 
            "name": topic_info["name"], 
            "start_index": topic_info["start"], 
            "end_index": topic_info["end"], 
            "num_chunks": topic_info["end"] - topic_info["start"] + 1
        }
        topics.append(topic_data)
    return {"total_topics": len(topics), "total_chunks": len(rag_system.chunks), "topics": topics}

@app.post("/generate_quiz")
async def generate_quiz_stream(request: QuizRequest):
    if request.num_questions < 1:
        raise HTTPException(status_code=400, detail="Số câu hỏi phải lớn hơn 0")
    
    topic_names = []
    if request.topic_ids:
        for tid in request.topic_ids:
            if tid not in TOPIC_RANGES:
                raise HTTPException(status_code=400, detail=f"Topic ID {tid} không tồn tại. Vui lòng kiểm tra lại.")
            topic_names.append(TOPIC_RANGES[tid]["name"])

    async def response_generator():
        try:
            display_topic_name = "Tổng hợp (Nhiều chủ đề)"
            if request.topic_ids:
                if len(request.topic_ids) == 1:
                    display_topic_name = TOPIC_RANGES[request.topic_ids[0]]["name"]
                else:
                    display_topic_name = f"Tổng hợp: {', '.join([str(tid) for tid in request.topic_ids])}"

            # Gọi hàm với list topic_ids
            iterator = quiz_generator.generate_stream(request.num_questions, request.topic_ids)
            
            for batch_result in iterator:
                chunk_data = {
                    "status": "processing",
                    "session_id": request.session_id,
                    "topic_name": display_topic_name,
                    "current_count": batch_result["generated_count"],
                    "total_requested": batch_result["total_requested"],
                    "new_questions": batch_result["batch_data"]
                }
                yield json.dumps(chunk_data, ensure_ascii=False) + "\n"
                await asyncio.sleep(0.01) 
            
            yield json.dumps({
                "status": "completed", 
                "session_id": request.session_id,
                "message": "Đã hoàn thành tạo câu hỏi"
            }, ensure_ascii=False) + "\n"

        except Exception as e:
            yield json.dumps({
                "status": "error",
                "session_id": request.session_id,
                "message": str(e)
            }, ensure_ascii=False) + "\n"

    return StreamingResponse(response_generator(), media_type="application/x-ndjson")

In [ ]:
import uvicorn
import nest_asyncio
from pyngrok import ngrok, conf
import socket
import threading
import time

nest_asyncio.apply()

NGROK_AUTH_TOKEN = ""

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

def get_free_port(start_port=8000):
    port = start_port
    while True:
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.bind(('0.0.0.0', port))
            sock.close()
            return port
        except OSError:
            port += 1

def start_ngrok_tunnel(port):
    try:
        conf.get_default().region = "us"
        # Đóng các tunnel cũ để tránh lỗi
        ngrok.kill()
        time.sleep(1)
        
        # Mở tunnel mới
        tunnel = ngrok.connect(port, proto="http", bind_tls=True)
        return tunnel
    except Exception as e:
        print(f"Ngrok error: {e}")
        return None

def run_server(port):
    config = uvicorn.Config(app, host="0.0.0.0", port=port, log_level="info")
    server = uvicorn.Server(config)
    server.run()

def start_server_with_ngrok():
    free_port = get_free_port(8000)
    
    server_thread = threading.Thread(target=run_server, args=(free_port,), daemon=True, name="FastAPI-Server")
    server_thread.start()
    
    time.sleep(3)
    tunnel = start_ngrok_tunnel(free_port)
    
    if tunnel:
        print(f" Server is running!")
        print(f" Public URL: {tunnel.public_url}")
    return server_thread, tunnel

server_thread, tunnel = start_server_with_ngrok()